## Шаг 1. Преобразование текста в байты

Беру фамилию «Гркикян» и кодирую в **CP1251** — кириллическая кодировка, где каждый символ занимает **1 байт**.



In [188]:
text = "Гркикян"
b = text.encode("cp1251")
print(b) # в шестнадцатиричном виде
print(list(b)) # в десятичном виде

b'\xc3\xf0\xea\xe8\xea\xff\xed'
[195, 240, 234, 232, 234, 255, 237]


In [189]:
IP = [
    58, 50, 42, 34, 26, 18, 10, 2,
    60, 52, 44, 36, 28, 20, 12, 4,
    62, 54, 46, 38, 30, 22, 14, 6,
    64, 56, 48, 40, 32, 24, 16, 8,
    57, 49, 41, 33, 25, 17,  9, 1,
    59, 51, 43, 35, 27, 19, 11, 3,
    61, 53, 45, 37, 29, 21, 13, 5,
    63, 55, 47, 39, 31, 23, 15, 7
] # Начальная перестановка (64 бита)

PC1 = [
    57, 49, 41, 33, 25, 17,  9,
     1, 58, 50, 42, 34, 26, 18,
    10,  2, 59, 51, 43, 35, 27,
    19, 11,  3, 60, 52, 44, 36,
    63, 55, 47, 39, 31, 23, 15,
     7, 62, 54, 46, 38, 30, 22,
    14,  6, 61, 53, 45, 37, 29,
    21, 13,  5, 28, 20, 12,  4
] # Сжатие ключа 64 → 56 бит

SHIFTS = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1] # Сдвиги C и D по раундам

PC2 = [
    14, 17, 11, 24,  1,  5,
     3, 28, 15,  6, 21, 10,
    23, 19, 12,  4, 26,  8,
    16,  7, 27, 20, 13,  2,
    41, 52, 31, 37, 47, 55,
    30, 40, 51, 45, 33, 48,
    44, 49, 39, 56, 34, 53,
    46, 42, 50, 36, 29, 32
] # Сжатие ключа 56 → 48 бит

E = [
    32,  1,  2,  3,  4,  5,
     4,  5,  6,  7,  8,  9,
     8,  9, 10, 11, 12, 13,
    12, 13, 14, 15, 16, 17,
    16, 17, 18, 19, 20, 21,
    20, 21, 22, 23, 24, 25,
    24, 25, 26, 27, 28, 29,
    28, 29, 30, 31, 32,  1
] # Расширение R с 32 → 48 бит

S_BOXES = [
    # S1
    [[14, 4, 13, 1, 2, 15, 11, 8, 3, 10, 6, 12, 5, 9, 0, 7],
     [0, 15, 7, 4, 14, 2, 13, 1, 10, 6, 12, 11, 9, 5, 3, 8],
     [4, 1, 14, 8, 13, 6, 2, 11, 15, 12, 9, 7, 3, 10, 5, 0],
     [15, 12, 8, 2, 4, 9, 1, 7, 5, 11, 3, 14, 10, 0, 6, 13]],
    # S2
    [[15, 1, 8, 14, 6, 11, 3, 4, 9, 7, 2, 13, 12, 0, 5, 10],
     [3, 13, 4, 7, 15, 2, 8, 14, 12, 0, 1, 10, 6, 9, 11, 5],
     [0, 14, 7, 11, 10, 4, 13, 1, 5, 8, 12, 6, 9, 3, 2, 15],
     [13, 8, 10, 1, 3, 15, 4, 2, 11, 6, 7, 12, 0, 5, 14, 9]],
    # S3
    [[10, 0, 9, 14, 6, 3, 15, 5, 1, 13, 12, 7, 11, 4, 2, 8],
     [13, 7, 0, 9, 3, 4, 6, 10, 2, 8, 5, 14, 12, 11, 15, 1],
     [13, 6, 4, 9, 8, 15, 3, 0, 11, 1, 2, 12, 5, 10, 14, 7],
     [1, 10, 13, 0, 6, 9, 8, 7, 4, 15, 14, 3, 11, 5, 2, 12]],
    # S4
    [[7, 13, 14, 3, 0, 6, 9, 10, 1, 2, 8, 5, 11, 12, 4, 15],
     [13, 8, 11, 5, 6, 15, 0, 3, 4, 7, 2, 12, 1, 10, 14, 9],
     [10, 6, 9, 0, 12, 11, 7, 13, 15, 1, 3, 14, 5, 2, 8, 4],
     [3, 15, 0, 6, 10, 1, 13, 8, 9, 4, 5, 11, 12, 7, 2, 14]],
    # S5
    [[2, 12, 4, 1, 7, 10, 11, 6, 8, 5, 3, 15, 13, 0, 14, 9],
     [14, 11, 2, 12, 4, 7, 13, 1, 5, 0, 15, 10, 3, 9, 8, 6],
     [4, 2, 1, 11, 10, 13, 7, 8, 15, 9, 12, 5, 6, 3, 0, 14],
     [11, 8, 12, 7, 1, 14, 2, 13, 6, 15, 0, 9, 10, 4, 5, 3]],
    # S6
    [[12, 1, 10, 15, 9, 2, 6, 8, 0, 13, 3, 4, 14, 7, 5, 11],
     [10, 15, 4, 2, 7, 12, 9, 5, 6, 1, 13, 14, 0, 11, 3, 8],
     [9, 14, 15, 5, 2, 8, 12, 3, 7, 0, 4, 10, 1, 13, 11, 6],
     [4, 3, 2, 12, 9, 5, 15, 10, 11, 14, 1, 7, 6, 0, 8, 13]],
    # S7
    [[4, 11, 2, 14, 15, 0, 8, 13, 3, 12, 9, 7, 5, 10, 6, 1],
     [13, 0, 11, 7, 4, 9, 1, 10, 14, 3, 5, 12, 2, 15, 8, 6],
     [1, 4, 11, 13, 12, 3, 7, 14, 10, 15, 6, 8, 0, 5, 9, 2],
     [6, 11, 13, 8, 1, 4, 10, 7, 9, 5, 0, 15, 14, 2, 3, 12]],
    # S8
    [[13, 2, 8, 4, 6, 15, 11, 1, 10, 9, 3, 14, 5, 0, 12, 7],
     [1, 15, 13, 8, 10, 3, 7, 4, 12, 5, 6, 11, 0, 14, 9, 2],
     [7, 11, 4, 1, 9, 12, 14, 2, 0, 6, 10, 13, 15, 3, 5, 8],
     [2, 1, 14, 7, 4, 10, 8, 13, 15, 12, 9, 0, 3, 5, 6, 11]]
] # 8 S-боксов для замены 6 бит → 4 бита

P = [
    16,  7, 20, 21, 29, 12, 28, 17,
     1, 15, 23, 26,  5, 18, 31, 10,
     2,  8, 24, 14, 32, 27,  3,  9,
    19, 13, 30,  6, 22, 11,  4, 25
] # Перестановка после S-боксов

FP = [
    40, 8, 48, 16, 56, 24, 64, 32,
    39, 7, 47, 15, 55, 23, 63, 31,
    38, 6, 46, 14, 54, 22, 62, 30,
    37, 5, 45, 13, 53, 21, 61, 29,
    36, 4, 44, 12, 52, 20, 60, 28,
    35, 3, 43, 11, 51, 19, 59, 27,
    34, 2, 42, 10, 50, 18, 58, 26,
    33, 1, 41,  9, 49, 17, 57, 25
]

In [190]:
text = "Гркикян"
key_text = "Мисак"

## Кодирование текста и ключа в байты:

In [191]:
data_cp1251 = text.encode("cp1251")
key_cp1251 = key_text.encode("cp1251")

In [192]:
print(data_cp1251) # 7 байт
print(key_cp1251) # 5 байт

b'\xc3\xf0\xea\xe8\xea\xff\xed'
b'\xcc\xe8\xf1\xe0\xea'


In [193]:
print("Text in cp1251:", data_cp1251.hex())
print("Key in cp1251:", key_cp1251.hex())

Text in cp1251: c3f0eae8eaffed
Key in cp1251: cce8f1e0ea


In [194]:
hex_text = data_cp1251.hex()
print(hex_text)
hex_key = key_cp1251.hex()
print(hex_key)

print(' '.join(format(int(c, 16), '04b') for c in hex_text))
print(' '.join(format(int(c, 16), '04b') for c in hex_key))

c3f0eae8eaffed
cce8f1e0ea
1100 0011 1111 0000 1110 1010 1110 1000 1110 1010 1111 1111 1110 1101
1100 1100 1110 1000 1111 0001 1110 0000 1110 1010


## Дополняем текст и ключ нулевыми байтами спереди:

In [195]:
print(len(data_cp1251)) # 7 байт - 7 символов в моей фамилии
list(b'\x00')
data_padded = b'\x00' * (8 - len(data_cp1251)) + data_cp1251
print(data_padded) # шифруемое слово

key_padded = b'\x00' * (8 - len(key_cp1251)) + key_cp1251
print(key_padded)
print(len(data_padded), len(key_padded))

7
b'\x00\xc3\xf0\xea\xe8\xea\xff\xed'
b'\x00\x00\x00\xcc\xe8\xf1\xe0\xea'
8 8


In [196]:
data_padded_hex = data_padded.hex()
print(data_padded_hex)
key_padded_hex = key_padded.hex()
print(key_padded_hex)

print(' '.join(format(int(c, 16), '04b') for c in data_padded_hex))
print(' '.join(format(int(c, 16), '04b') for c in key_padded_hex))

print(len(data_padded_hex))

00c3f0eae8eaffed
000000cce8f1e0ea
0000 0000 1100 0011 1111 0000 1110 1010 1110 1000 1110 1010 1111 1111 1110 1101
0000 0000 0000 0000 0000 0000 1100 1100 1110 1000 1111 0001 1110 0000 1110 1010
16


In [197]:
bits_str = ''.join(format(int(c, 16), '04b') for c in data_padded_hex)
print(bits_str)

print("\nДанные - 64 бита:")
for i, bit in enumerate(bits_str, start=1):
    print(f"{i:2}:{bit}", end="  ")
    if i % 8 == 0:
        print()

0000000011000011111100001110101011101000111010101111111111101101

Данные - 64 бита:
 1:0   2:0   3:0   4:0   5:0   6:0   7:0   8:0  
 9:1  10:1  11:0  12:0  13:0  14:0  15:1  16:1  
17:1  18:1  19:1  20:1  21:0  22:0  23:0  24:0  
25:1  26:1  27:1  28:0  29:1  30:0  31:1  32:0  
33:1  34:1  35:1  36:0  37:1  38:0  39:0  40:0  
41:1  42:1  43:1  44:0  45:1  46:0  47:1  48:0  
49:1  50:1  51:1  52:1  53:1  54:1  55:1  56:1  
57:1  58:1  59:1  60:0  61:1  62:1  63:0  64:1  


In [198]:
data_bits = []
for b in bits_str:
    data_bits.append(b)

print(data_bits)
type(data_bits[8])

['0', '0', '0', '0', '0', '0', '0', '0', '1', '1', '0', '0', '0', '0', '1', '1', '1', '1', '1', '1', '0', '0', '0', '0', '1', '1', '1', '0', '1', '0', '1', '0', '1', '1', '1', '0', '1', '0', '0', '0', '1', '1', '1', '0', '1', '0', '1', '0', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '0', '1', '1', '0', '1']


str

In [199]:
# перестановка
def permute(bits, table):
    result = []
    for i in table:
        result.append(bits[i - 1])
    return result

In [200]:
data_after_ip = permute(data_bits, IP)
print(data_after_ip)

['1', '1', '1', '1', '1', '1', '1', '0', '0', '1', '0', '0', '0', '1', '0', '0', '1', '1', '0', '0', '0', '0', '0', '0', '1', '1', '0', '0', '0', '0', '1', '0', '1', '1', '1', '1', '1', '1', '1', '0', '1', '1', '1', '1', '1', '1', '0', '0', '1', '1', '1', '1', '1', '0', '0', '0', '0', '1', '1', '0', '1', '0', '1', '0']


In [201]:
bits_joined = ''.join(data_after_ip)
print(bits_joined)

result_hex = ''
for i in range(0, len(bits_joined), 8):
    byte_bits = bits_joined[i:i+8]
    byte_value = int(byte_bits, 2)
    result_hex += format(byte_value, '02x')

print("Hex: ", result_hex)

1111111001000100110000001100001011111110111111001111100001101010
Hex:  fe44c0c2fefcf86a


In [202]:
L0 = data_after_ip[:32]
R0 = data_after_ip[32:]
print("L0: ", L0)
print("R0: ", R0)
print(len(L0))
print(len(R0))

L0:  ['1', '1', '1', '1', '1', '1', '1', '0', '0', '1', '0', '0', '0', '1', '0', '0', '1', '1', '0', '0', '0', '0', '0', '0', '1', '1', '0', '0', '0', '0', '1', '0']
R0:  ['1', '1', '1', '1', '1', '1', '1', '0', '1', '1', '1', '1', '1', '1', '0', '0', '1', '1', '1', '1', '1', '0', '0', '0', '0', '1', '1', '0', '1', '0', '1', '0']
32
32


In [203]:
hex_key_str = key_padded.hex()

key_bits_str = ''.join(format(int(c, 16), '04b') for c in hex_key_str)
print(key_bits_str)

print("Ключ - 64 бита:")
for i, bit in enumerate(key_bits_str, start=1):
    print(f"{i:2}:{bit}", end="  ")
    if i % 8 == 0:
        print()

0000000000000000000000001100110011101000111100011110000011101010
Ключ - 64 бита:
 1:0   2:0   3:0   4:0   5:0   6:0   7:0   8:0  
 9:0  10:0  11:0  12:0  13:0  14:0  15:0  16:0  
17:0  18:0  19:0  20:0  21:0  22:0  23:0  24:0  
25:1  26:1  27:0  28:0  29:1  30:1  31:0  32:0  
33:1  34:1  35:1  36:0  37:1  38:0  39:0  40:0  
41:1  42:1  43:1  44:1  45:0  46:0  47:0  48:1  
49:1  50:1  51:1  52:0  53:0  54:0  55:0  56:0  
57:1  58:1  59:1  60:0  61:1  62:0  63:1  64:0  


In [204]:
key_bits = []
for b in key_bits_str:
    key_bits.append(b)

print(key_bits)
key_bits[10]

['0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '1', '0', '0', '1', '1', '0', '0', '1', '1', '1', '0', '1', '0', '0', '0', '1', '1', '1', '1', '0', '0', '0', '1', '1', '1', '1', '0', '0', '0', '0', '0', '1', '1', '1', '0', '1', '0', '1', '0']


'0'

In [205]:
key_pc1 = permute(key_bits, PC1)
print(key_pc1)
print(len(key_pc1))

['1', '1', '1', '1', '1', '0', '0', '0', '1', '1', '1', '1', '1', '0', '0', '0', '1', '1', '1', '1', '0', '0', '0', '0', '0', '0', '1', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '1', '0', '0', '1', '1', '0', '0', '0', '0', '0', '0', '0']
56


In [206]:
C = key_pc1[:28]
D = key_pc1[28:]
print(len(C))
print(len(D))

28
28


In [207]:
def left_shift(bits, n):
    n = n % len(bits) # на сколько сдвигаем
    return bits[n:] + bits[:n]

In [208]:
C1 = left_shift(C, 1)
D1 = left_shift(D, 1)
print(C1)
print(D1)

['1', '1', '1', '1', '0', '0', '0', '1', '1', '1', '1', '1', '0', '0', '0', '1', '1', '1', '1', '0', '0', '0', '0', '0', '0', '1', '0', '1']
['0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '1', '0', '0', '1', '1', '0', '0', '0', '0', '0', '0', '0', '1']


In [209]:
CD1 = C1 + D1
print(len(CD1))

56


In [210]:
K1 = permute(CD1, PC2)
print(K1) # ключ 1-го раунда
print(len(K1))

['0', '1', '1', '0', '1', '0', '1', '1', '0', '0', '0', '1', '0', '1', '1', '1', '1', '1', '1', '0', '0', '0', '0', '1', '0', '0', '0', '0', '1', '0', '0', '1', '0', '0', '0', '1', '1', '0', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0']
48


In [211]:
K1_str = ''.join(K1)

K1_hex = ''
for i in range(0, len(K1_str), 4):
    nibble = K1_str[i:i+4]
    value = int(nibble, 2)
    K1_hex += format(value, 'x')

print("Hex: ", K1_hex)
print(len(K1_hex))

Hex:  6b17e1091900
12


In [212]:
print(R0)

print("R0 — 32 бита:")
for i, bit in enumerate(R0, start=1):
    print(f"{i:2}:{bit}", end="  ")
    if i % 8 == 0:
        print()

['1', '1', '1', '1', '1', '1', '1', '0', '1', '1', '1', '1', '1', '1', '0', '0', '1', '1', '1', '1', '1', '0', '0', '0', '0', '1', '1', '0', '1', '0', '1', '0']
R0 — 32 бита:
 1:1   2:1   3:1   4:1   5:1   6:1   7:1   8:0  
 9:1  10:1  11:1  12:1  13:1  14:1  15:0  16:0  
17:1  18:1  19:1  20:1  21:1  22:0  23:0  24:0  
25:0  26:1  27:1  28:0  29:1  30:0  31:1  32:0  


In [213]:
R_expanded = permute(R0, E)
print(R_expanded)
print(len(R_expanded))

['0', '1', '1', '1', '1', '1', '1', '1', '1', '1', '0', '1', '0', '1', '1', '1', '1', '1', '1', '1', '1', '0', '0', '1', '0', '1', '1', '1', '1', '1', '1', '1', '0', '0', '0', '0', '0', '0', '1', '1', '0', '1', '0', '1', '0', '1', '0', '1']
48


In [214]:
print(R_expanded)
print(K1)
print(len(R_expanded))
print(len(K1))
print(type(R_expanded))
print(type(K1))

['0', '1', '1', '1', '1', '1', '1', '1', '1', '1', '0', '1', '0', '1', '1', '1', '1', '1', '1', '1', '1', '0', '0', '1', '0', '1', '1', '1', '1', '1', '1', '1', '0', '0', '0', '0', '0', '0', '1', '1', '0', '1', '0', '1', '0', '1', '0', '1']
['0', '1', '1', '0', '1', '0', '1', '1', '0', '0', '0', '1', '0', '1', '1', '1', '1', '1', '1', '0', '0', '0', '0', '1', '0', '0', '0', '0', '1', '0', '0', '1', '0', '0', '0', '1', '1', '0', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0']
48
48
<class 'list'>
<class 'list'>


XOR - если биты одинаковые, то 0. Если разные, то 1.

In [215]:
result = [str(int(a) ^ int(b)) for a, b in zip(R_expanded, K1)]
print(result)

['0', '0', '0', '1', '0', '1', '0', '0', '1', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '1', '0', '0', '0', '0', '1', '1', '1', '0', '1', '1', '0', '0', '0', '0', '1', '1', '0', '1', '0', '0', '1', '0', '1', '0', '1', '0', '1']


In [216]:
bits_48 = ''.join(result)
print(len(bits_48))

print(' '.join(bits_48[i:i+6] for i in range(0, len(bits_48), 6)))

48
000101 001100 000000 011000 011101 100001 101001 010101


In [217]:
for i in range(0, 48, 6):
    group = bits_48[i:i+6]
    box_num = i // 6 + 1

    row_str = group[0] + group[5]
    row = int(row_str, 2)

    col_str = group[1:5]
    col = int(col_str, 2)

    print(f"S{box_num}: {group} ->  строка: {row} ({row_str}), столбец: {col} ({col_str})")

S1: 000101 ->  строка: 1 (01), столбец: 2 (0010)
S2: 001100 ->  строка: 0 (00), столбец: 6 (0110)
S3: 000000 ->  строка: 0 (00), столбец: 0 (0000)
S4: 011000 ->  строка: 0 (00), столбец: 12 (1100)
S5: 011101 ->  строка: 1 (01), столбец: 14 (1110)
S6: 100001 ->  строка: 3 (11), столбец: 0 (0000)
S7: 101001 ->  строка: 3 (11), столбец: 4 (0100)
S8: 010101 ->  строка: 1 (01), столбец: 10 (1010)


In [218]:
print(len(S_BOXES))
print(len(S_BOXES[0]))
print(len(S_BOXES[0][0]))

8
4
16


In [219]:
for i in range(0, 48, 6):
    group = bits_48[i:i+6]
    box_num = i // 6 + 1

    row_str = group[0] + group[5]
    row = int(row_str, 2)

    col_str = group[1:5]
    col = int(col_str, 2)

    value = S_BOXES[box_num - 1][row][col] # 16-ричная система (4 бита)

    print(f"S{box_num}: {group} ->  строка: {row}, столбец: {col}, value: {value}")

S1: 000101 ->  строка: 1, столбец: 2, value: 7
S2: 001100 ->  строка: 0, столбец: 6, value: 3
S3: 000000 ->  строка: 0, столбец: 0, value: 10
S4: 011000 ->  строка: 0, столбец: 12, value: 11
S5: 011101 ->  строка: 1, столбец: 14, value: 8
S6: 100001 ->  строка: 3, столбец: 0, value: 4
S7: 101001 ->  строка: 3, столбец: 4, value: 1
S8: 010101 ->  строка: 1, столбец: 10, value: 6


In [220]:
print(type(value))

<class 'int'>


In [221]:
s_output = []

for i in range(0, 48, 6):
    group = bits_48[i:i+6]
    box_num = i // 6 + 1

    row_str = group[0] + group[5]
    row = int(row_str, 2)

    col_str = group[1:5]
    col = int(col_str, 2)

    value = S_BOXES[box_num - 1][row][col]

    bits4 = format(value, '04b')

    print(f"S{box_num}: {group} ->  строка: {row}, столбец: {col}, value: {value} -> {bits4}")

S1: 000101 ->  строка: 1, столбец: 2, value: 7 -> 0111
S2: 001100 ->  строка: 0, столбец: 6, value: 3 -> 0011
S3: 000000 ->  строка: 0, столбец: 0, value: 10 -> 1010
S4: 011000 ->  строка: 0, столбец: 12, value: 11 -> 1011
S5: 011101 ->  строка: 1, столбец: 14, value: 8 -> 1000
S6: 100001 ->  строка: 3, столбец: 0, value: 4 -> 0100
S7: 101001 ->  строка: 3, столбец: 4, value: 1 -> 0001
S8: 010101 ->  строка: 1, столбец: 10, value: 6 -> 0110


In [222]:
s_output_str = ''
for i in range(0, 48, 6):
    group = bits_48[i:i+6]
    box_num = i // 6 + 1

    row_str = group[0] + group[5]
    row = int(row_str, 2)

    col_str = group[1:5]
    col = int(col_str, 2)

    value = S_BOXES[box_num - 1][row][col]

    s_output_str += format(value, '04b')

print("S-выход (32 бита): ", s_output_str)
print(type(s_output_str))
print(len(s_output_str))
# print(type(int(s_output_str)))

S-выход (32 бита):  01110011101010111000010000010110
<class 'str'>
32


In [223]:
print("S-выход - 32 бита:")
for i, bit in enumerate(s_output_str, start=1):
  print(f"{i:2}:{bit}", end="  ")
  if i % 8 == 0:
    print()

S-выход - 32 бита:
 1:0   2:1   3:1   4:1   5:0   6:0   7:1   8:1  
 9:1  10:0  11:1  12:0  13:1  14:0  15:1  16:1  
17:1  18:0  19:0  20:0  21:0  22:1  23:0  24:0  
25:0  26:0  27:0  28:1  29:0  30:1  31:1  32:0  


In [224]:
print(s_output_str)

s_output = [int(b) for b in s_output_str]
print(s_output)

f_result = permute(s_output, P)
print(f_result)

01110011101010111000010000010110
[0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0]
[1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0]


In [225]:
f_str = ''.join(str(b) for b in f_result)
print(f_str)
print(type(f_str))

print(' '.join(f_str[i:i+4] for i in range(0, 32, 4)))

11000011010000101100001101101110
<class 'str'>
1100 0011 0100 0010 1100 0011 0110 1110


In [226]:
f_hex = ''
for i in range(0, 32, 4):
  nibble = f_str[i:i+4]
  value = int(nibble, 2)
  f_hex += format(value, 'x')

print("Hex: ", f_hex)

Hex:  c342c36e


In [227]:
print(L0)

['1', '1', '1', '1', '1', '1', '1', '0', '0', '1', '0', '0', '0', '1', '0', '0', '1', '1', '0', '0', '0', '0', '0', '0', '1', '1', '0', '0', '0', '0', '1', '0']


In [228]:
L0_int = [int(b) for b in L0]
print(L0_int)
print(f_result)
# print(len(L0_int))
# print(len(f_result))

R1 = [a ^ b for a, b in zip(L0_int, f_result)]
print(R1)
print(len(R1))

[1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0]
[1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0]
[0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0]
32


In [229]:
R1_str = ''.join(str(b) for b in R1)

R1_hex = ''
for i in range(0, 32, 4):
  nibble = R1_str[i:i+4]
  value = int(nibble, 2)
  R1_hex += format(value, 'x')

print("Hex: ", R1_hex)

Hex:  3d0603ac


In [230]:
print(R0)
print(type(R0[0]))

['1', '1', '1', '1', '1', '1', '1', '0', '1', '1', '1', '1', '1', '1', '0', '0', '1', '1', '1', '1', '1', '0', '0', '0', '0', '1', '1', '0', '1', '0', '1', '0']
<class 'str'>


In [231]:
R0_str = ''.join(R0)
print(R0_str)

R0_hex = ''
for i in range(0, 32, 4):
  nibble = R0_str[i:i+4]
  value = int(nibble, 2)
  R0_hex += format(value, 'x')

print("Hex: ", R0_hex)

11111110111111001111100001101010
Hex:  fefcf86a


In [232]:
print(R0_hex + R1_hex) # блок после 1 раунда

fefcf86a3d0603ac


In [233]:
L = data_after_ip[:32]
R = data_after_ip[32:]
C = key_pc1[:28]
D = key_pc1[28:]

for round_num in range(16):
    shift = SHIFTS[round_num]
    C = left_shift(C, shift)
    D = left_shift(D, shift)

    CD = C + D
    K = permute(CD, PC2)

    R_expanded = permute(R, E)
    xor_result = [str(int(a) ^ int(b)) for a, b in zip(R_expanded, K)]
    bits_48 = ''.join(xor_result)

    s_output = []
    for i in range(0, 48, 6):
        group = bits_48[i:i+6]
        box_num = i // 6 + 1
        row = int(group[0] + group[5], 2)
        col = int(group[1:5], 2)
        value = S_BOXES[box_num - 1][row][col]
        s_output.append(value)

    s_output_str = ''.join(format(v, '04b') for v in s_output)

    s_output_bits = [int(b) for b in s_output_str]
    f_result = permute(s_output_bits, P)

    L_int = [int(b) for b in L]
    R_new = [a ^ b for a, b in zip(L_int, f_result)]
    L, R = R, R_new

final_block = R + L
encrypted_bits = permute(final_block, FP)